# OT-2: Mean-Line Turbine Design

> Author: Elias Aoubala

> Date: 22/12/2025

In [228]:
from turborocket.meanline.meanline_relations import TurbineStageDesign
from turborocket.fluids.fluids import IdealGas

import numpy as np
import pandas as pd

import yaml

import handcalcs.render

## 1 - Background

This document encapsulates the mean-line design of the turbine-stage of the `OT-2: Man-Ray and the Dirty Bubble`. A high level mean line design study is conducted using our in-house python package `turboRocket`, from which an iterative optimisation is performed to evaluate for our key gas-generator mass flows needed to meet our power requirements.

## 2 - High Level Functional Parmaeters (as loaded from yaml file)

We have now refactored our code so the input files are stored in `yaml` files for both the gas-generator characteristics and the turbine user selected inputs.

Hence, we just load them in via yaml files.

In [354]:
meanline_directory = "meanline"

### 2.1 - Gas Generator Inputs

In [357]:
with open(f"{meanline_directory}/gg_out.yaml") as f:

    gg_inputs = yaml.safe_load(f)

We can extract our gas object properties accordingly.

In [358]:
gas_dic = gg_inputs["gas_generator"]["operating_point"]["gas"]

fluid = IdealGas(
    p=gas_dic["P"],
    t=gas_dic["T"],
    gamma=gas_dic["gamma"],
    cp=gas_dic["Cp"],
    R=gas_dic["R"],
)

And we can extract our geometries for our nozzle throat and mass flow.

In [359]:
gg_m_dot = gg_inputs["gas_generator"]["operating_point"]["m_dot"]

gg_geom = gg_inputs["gas_generator"]["geometry"]["chamber"]

### 2.2 - Turbine Inputs

In [360]:
with open(f"{meanline_directory}/turbine_in.yaml") as f:

    turbine_input = yaml.safe_load(f)

We can extract our key parameters, in this case required power, stator parameters and rotor

In [361]:
turbine_rotor = turbine_input["turbine_stage"]["rotor"]
turbine_stator = turbine_input["turbine_stage"]["stator"]

turbine_power = turbine_input["turbine_stage"]["power"]

## 4 - Mean-Line Design Power Optimization

Our current approach for mean-line design does not design to user specification, but rather takes in some geometric parameters from the user, and calculates the expected turbine stage performance from it (in this case power, etc). As we have a target shaft power we must hit, this setions aims to conduct a primitive optimisation where the gas-generator mass flow is iterated on, until the target shaft power is met.

### 4.1 - Inital Parameter Guesses

As this problem is an optimisation problem, we will manually guess the blade spacing and thickness. These will updated once the turbine profile has been designed.

In [362]:
b = 15e-3
t = 5e-3

Based on these inputs, we can derive the required turbine power accordingly - using our refined loss model.

The main parameter we will iterate on the GG mass flow rate untill we reach our target power. We will do an adjoint-esque iterative study until the power requirement is reached.

We will instantiate an inital error of 0

In [363]:
error = 0

### 4.2 - Optimisation Loop for System Power

We will use a relaxation factor of **0.4** for the adjoint solver based on the relative error from the target power.

In [408]:
relax = 0.4

In [409]:
print(f"Blade Chord Length: {b*1e3}")
print(f"Blade Spacing: {t*1e3}")
print(turbine_rotor["N_shaft"])
m_dot_t = gg_m_dot["total"] 

Blade Chord Length: 15.0
Blade Spacing: 5.0
30000.0


In [411]:
m_dot_t -= m_dot_t*error*relax

stage = TurbineStageDesign(gas=fluid, m_dot=m_dot_t, omega=turbine_rotor["N_shaft"], alpha=(90 - turbine_stator["alpha"]))

stage.set_operating_point(u_cis=turbine_rotor["u_cis"], Rt=turbine_stator["Rt"], b=b, t=t, delta_r=turbine_rotor["delta_r"], N=turbine_stator["N_nozzle"])

result = stage.solve_performance(phi_n=turbine_stator["phi_n"])

P = result["performance"]["Power"]

error = (P - turbine_power) / turbine_power

print(f"Relative Error: {error*1e2:.2f} %")

Current Error: 8.815972518840862 %
Current Error: 0.7309485405195234 %
Current Error: 0.06525983563990781 %
Current Error: 0.005789389065834222 %
Relative Error: -9.84 %


We can get out power produced and calculate our error to repeat once again.

**Final Gas Generator Mass Flow Rate**

In [414]:
%%render param

m= m_dot_t*1000

<IPython.core.display.Latex object>

## 5 - Expected Mean-Line Performance Metrics

We can now get an idea of the specific power of the turbine and the associated performance Metrics

In [336]:
performance_dic = result["performance"]
performance_dic = {k: [v] for k, v in performance_dic.items()}
performance_df = pd.DataFrame(performance_dic)

pressure_dic = result["pressure"]
pressure_dic = {k: [v] for k, v in pressure_dic.items()}
pressure_df = pd.DataFrame(pressure_dic)

velocity_dic = result["velocity"]
velocity_dic = {k: [v] for k, v in velocity_dic.items()}
velocity_df = pd.DataFrame(velocity_dic)

temperature_dic = result["temperature"]
temperature_dic = {k: [v] for k, v in temperature_dic.items()}
temperature_df = pd.DataFrame(temperature_dic)

geometry_dic = result["geometry"]
geometry_dic = {k: [v] for k, v in geometry_dic.items()}
geometry_df = pd.DataFrame(geometry_dic)

mach_dic = result["mach"]
mach_dic = {k: [v] for k, v in mach_dic.items()}
mach_df = pd.DataFrame(mach_dic)

angles_dic = result["angles"]
angles_dic = {k: [v] for k, v in angles_dic.items()}
angles_df = pd.DataFrame(angles_dic)

### 5.1 - High Level Functional Parameters

In [337]:
performance_df["Power (kW)"] = performance_df["Power"].multiply(1e-3)
performance_df["Torque (Nm)"] = performance_df["Power"] / (25000 * 2 *np.pi/60)

performance_df

,dh,eps,phi_r,phi_l,m_leakage,eta_l,phi,eta_h,zeta_eps,eta_o,Power,Power (kW),Torque (Nm)
0,985358.187609,0.405949,0.781978,0.252982,0.028322,0.747018,0.85,0.247605,0.002119,0.182847,20170.716772,20.170717,7.704646


As can be seen, we are getting an efficiency on the order of **45%** with a specific power of 987 kW/kg/s

This works out to a GG size of the following:

### 5.2 - Expected Stage Pressures

These are the expected stage pressures in **Bar**.

In [338]:
pressure_df * 1e-5

,p_0,p_1,p_1o,p_1o_r,p_2o_r,p_2o
0,25.0,1.086957,8.201209,5.775221,2.789925,2.194535


### 5.3 - Expected Stage Temperatures

In [339]:
temperature_df

,t_0,t_1,t_1o_r,t_2,t_2o
0,868.169413,550.33941,802.294913,648.226886,759.247004


### 5.4 - Expected Stage Geometric Parameters

Here are the expected stage geometric parameters. We make a comparison between that calculated using the mean-line method, and that evaluated from our GG combustion solver to confirm the areas are comparable.

In [340]:
geometry_df["Nozzle Throat (mm)"] = [(gg_geom["A_cc"] / (turbine_stator["N_nozzle"]* np.pi))**(1/2) * 2 * 1e3]
geometry_df["Nozzle Exit (mm)"] = geometry_df["s_c"]*1e3
geometry_df["eps"] = geometry_df["A_1"] / geometry_df["A_0"]

throat_error = (geometry_df["A_0"][0] - gg_geom["A_cc"] )/ gg_geom["A_cc"]

print(f"Comparison of Throat Calculated using GG combustion solver: {throat_error*100:.1f} %")

geometry_df

Comparison of Throat Calculated using GG combustion solver: -0.1 %


,D_m,A_1,A_0,s_c,s_b,D_hub,D_tip,AR,Nozzle Throat (mm),Nozzle Exit (mm),eps
0,0.08937,0.000239,0.000045,0.006173,0.008673,0.080697,0.098043,0.578193,2.662345,6.172895,5.378913


### 5.5 - Expected Mach Numbers at Each Stage

In [341]:
mach_df

,m_star_c1,m_star_w1,m_star_w2,m_star_c2
0,1.699405,1.573971,1.23081,1.004385


### 5.6 - Expected Angles

For these angles, we subtracting from 90 to get the relative to the axial direction.

In [342]:
90 -angles_df

,beta_1,beta_2,alpha_2
0,67.409767,89.575543,54.49158


### 5.7 - Expected Stage Velocities

These are the expected absolute and relative velocities passing through the stage.

In [343]:
velocity_df

,u,c_1s,c_1,w_1,a_star_2,w_2,c_2,a_star_3
0,140.382206,1403.82206,1193.248751,1062.418113,674.99227,830.787212,705.235857,656.633935


## 6 - Exporting our Results

Now that we have done the high level mean-line design, we can extract our key parameters derived from the study, that we can place in a data fram and export as a yaml file, that can be read.

In [423]:
performance_out = { "performance": {k: float(v[0]) for k, v in performance_df.items()} }

pressure_out = { "pressure": {k: float(v[0]) for k, v in pressure_df.items()} }

velocity_out = { "velocity": {k: float(v[0]) for k, v in velocity_df.items()} }

temperature_out = { "temperature": {k: float(v[0]) for k, v in temperature_df.items()} }

geometry_out = { "geometry": {k: float(v[0]) for k, v in geometry_df.items()} }

mach_out = { "mach": {k: float(v[0]) for k, v in mach_df.items()} }

angles_out = { "angles": {k: float(v[0]) for k, v in angles_df.items()} }

mass_flow_out = {"error": {"input_val": float(gg_m_dot["total"]), "solved": float(m_dot_t), "m_dot_error": float((m_dot_t - gg_m_dot["total"])*100/gg_m_dot["total"]), "power_error": float(error)*100}}

input_params = {"input": turbine_input["turbine_stage"]}


We can now save this output file accordingly.

In [424]:
with open(f"{meanline_directory}/turbine_out.yaml", "w") as f:
    
    # Hight Level Title
    f.write(f"#"*80 + "\n")
    f.write(f"#{"OT-2: Man-Ray and The Dirty Bubble":^78}#\n")
    f.write(f"#{f"{"":-^40}":^78}#\n")
    f.write(f"#{"Meanline Design Study Results":^78}#\n")
    f.write(f"#{"":^78}#\n")
    f.write(f"#"*80 + "\n")
    f.write(f"\n\n")
    
    # Solution Convergence
    f.write(f"#"*80 + "\n")
    f.write(f"#{"Convergence Errors on Mass Flows":-^78}#\n")
    f.write(f"#"*80 + "\n")
    yaml.dump(mass_flow_out, f, default_flow_style=False)
    f.write(f"\n")
    
    # Input Parameters
    f.write(f"#"*80 + "\n")
    f.write(f"#{" Input Parameters ":-^78}#\n")
    f.write(f"#"*80 + "\n")
    yaml.dump(input_params, f, default_flow_style=False)
    f.write(f"\n")
    
    
    # Performance Results
    f.write(f"#"*80 + "\n")
    f.write(f"#{" Performance Results ":-^78}#\n")
    f.write(f"#"*80 + "\n")
    yaml.dump(performance_out, f, default_flow_style=False)
    f.write(f"\n")
    
    # Pressure Results
    f.write(f"#"*80 + "\n")
    f.write(f"#{" Pressure Results ":-^78}#\n")
    f.write(f"#"*80 + "\n")
    yaml.dump(pressure_out, f, default_flow_style=False)
    f.write(f"\n")
    
    # Velocity Results
    f.write(f"#"*80 + "\n")
    f.write(f"#{" Velocity Results ":-^78}#\n")
    f.write(f"#"*80 + "\n")
    yaml.dump(velocity_out, f, default_flow_style=False)
    f.write(f"\n")
    
    # Temperature Results
    f.write(f"#"*80 + "\n")
    f.write(f"#{" Temperature Results ":-^78}#\n")
    f.write(f"#"*80 + "\n")
    yaml.dump(temperature_out, f, default_flow_style=False)
    f.write(f"\n")
    
    # Geometry Results
    f.write(f"#"*80 + "\n")
    f.write(f"#{" Geometry Results ":-^78}#\n")
    f.write(f"#"*80 + "\n")
    yaml.dump(geometry_out, f, default_flow_style=False)
    f.write(f"\n")
    
    # Mach Results
    f.write(f"#"*80 + "\n")
    f.write(f"#{" Mach Results ":-^78}#\n")
    f.write(f"#"*80 + "\n")
    yaml.dump(mach_out, f, default_flow_style=False)
    f.write(f"\n")
    
    # Angles Results
    f.write(f"#"*80 + "\n")
    f.write(f"#{" Angles Results ":-^78}#\n")
    f.write(f"#"*80 + "\n")
    yaml.dump(angles_out, f, default_flow_style=False)
    f.write(f"\n")
    
    
